## Dataset & DataLoader
dataset-preprocessing.ipynb 에서 저장한 JSON / tokenizer 를 불러와서 Pytorch Dataset -> DataLoader 까지 구성

#### 0. 임포트

In [3]:
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

In [4]:
print(f"Pytorch: {torch.__version__}")
print(f"MPS 사용 가능: {torch.backends.mps.is_available()}")

Pytorch: 2.5.1
MPS 사용 가능: True


#### 1. 설정

In [6]:
DATA_DIR = Path("data")
MAX_LEN = 128
BATCH_SIZE = 32

PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

#### 2. Tokenizer 로드

In [8]:
tokenizer_file_path = str(DATA_DIR / "tokenizer" / "tokenizer.json")
tokenizer = Tokenizer.from_file(tokenizer_file_path)

VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"Vocab size : {VOCAB_SIZE}")

Vocab size : 16000


#### 3. Json 로드

In [10]:
def load_split(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_split(DATA_DIR / "train.json")
val_data = load_split(DATA_DIR / "val.json")
test_data = load_split(DATA_DIR / "test.json")

print(f"Train 데이터 크기: {len(train_data)}")
print(f"Validation 데이터 크기: {len(val_data)}")
print(f"Test 데이터 크기: {len(test_data)}")

Train 데이터 크기: 69300
Validation 데이터 크기: 8662
Test 데이터 크기: 8663


#### 4. TranslationDataset

In [12]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item["src"], item["tgt"]

In [13]:
train_ds = TranslationDataset(train_data)
val_ds = TranslationDataset(val_data)
test_ds = TranslationDataset(test_data)

# 단건 확인
src, tgt = train_ds[0]
print(f"src len: {len(src)}, tgt len: {len(tgt)}")

src len: 10, tgt len: 17


#### 5. collate_fn 정의 (배치 패딩)

In [15]:
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    def pad(seqs):
        padded = [s[:MAX_LEN] + [PAD_ID] * (MAX_LEN - len(s)) for s in seqs]

        return torch.tensor(padded, dtype=torch.long)

    return pad(src_batch), pad(tgt_batch)

#### 6. DataLoader

In [17]:
train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

test_dl = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

#### 7. Batch Shape 확인

In [19]:
src_batch, tgt_batch = next(iter(train_dl))

print(f"src: {src_batch.shape}, dtype={src_batch.dtype}")
print(f"tgt: {tgt_batch.shape}, dtype={tgt_batch.dtype}")

src: torch.Size([32, 128]), dtype=torch.int64
tgt: torch.Size([32, 128]), dtype=torch.int64


#### 8. Paddig mask 생성 (for transformer input)

In [21]:
def make_pad_mask(seq: torch.Tensor, pad_id: int = PAD_ID) -> torch.Tensor:
    """(B, 1, 1, T) - multi-head attention 브로드캐스팅 용도"""
    return (seq == pad_id).unsqueeze(1).unsqueeze(2)

def mask_casual_mask(seq_len: int, device: str = "cpu") -> torch.Tensor:
    """(1, 1, T, T) - 디코더 self-attention 인과 마스크 (상삼각 행렬로 구성해서 True 이면 미래 토큰을 의미)"""
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(1)

In [22]:
# 예시
src_batch, tgt_batch = next(iter(train_dl))

example_pad_mask = make_pad_mask(src_batch)
example_casual_mask = mask_casual_mask(MAX_LEN)

print(example_pad_mask.shape)
print(example_casual_mask.shape)

torch.Size([32, 1, 1, 128])
torch.Size([1, 1, 128, 128])


## Transformer 모델 구성

#### 0. 임포트 & 설정

In [25]:
import math, torch, torch.nn as nn, torch.nn.functional as F
from typing import Optional

In [26]:
VOCAB_SIZE = tokenizer.get_vocab_size()
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 4
D_FF = 1024
DROPOUT = 0.1
MAX_LEN = 128
PAD_ID = 0

In [27]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")

DEVICE = get_device()

print(DEVICE)

mps


#### 1. Positional Embedding

In [29]:
class PositionalEncoding(nn.Module):
    """PE(pos, 2i) = sin(pos / 10000^(2i/d_model))"""
    def __init__(self, d_model: int = D_MODEL, max_len: int = MAX_LEN, dropout: float = DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        # 분자 값
        pos = torch.arange(max_len).unsqueeze(1).float()
        # 분모 값
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
                * (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe.unsqueeze(0)) # (1, T, D) - Batch 차원을 더해야만 broadcast 가능

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [30]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size: int = VOCAB_SIZE, d_model: int = D_MODEL):
        super().__init__()
        
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.scale = math.sqrt(d_model)

    # (B, T) -> (B, T, D_MODEL)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.emb(x) * self.scale

In [31]:
# 임베딩 테스트
src_batch, tgt_batch = next(iter(train_dl))

embed = InputEmbedding()
positional_embed = PositionalEncoding()

input_embed_example = embed(src_batch)
print(input_embed_example.shape)

positional_embed_example = positional_embed(input_embed_example)
print(positional_embed_example.shape)

torch.Size([32, 128, 256])
torch.Size([32, 128, 256])


#### 2. Multi-Head Attention

In [33]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, dropout: float = DROPOUT):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.head_dim = d_model // n_heads
        self.scale = math.sqrt(self.head_dim)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B = q.size(0)
    
        # 1. Q, K, V projection (B, T, DIM_MODEL) -> (B, T, DIM_MODEL)
        q_proj = self.W_q(q)
        k_proj = self.W_k(k)
        v_proj = self.W_v(v)
    
        # 2. split heads (B, T, HEAD_DIM) -> (B, T, H, HEAD_DIM) -> (B, H, T, HEAD_DIM)
        Q = q_proj.view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        K = k_proj.view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        V = v_proj.view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
    
        # 3. attention score (B, H, T, HEAD_DIM) * (B, H, HEAD_DIM, T) -> (B, H, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
    
        # 4. mask
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
    
        # 5. attention probability
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
    
        # 6. weighted sum (B, H, T, T) * (B, H, T, HEAD_DIM) -> (B, H, T, HEAD_DIM)
        context = torch.matmul(attn, V)
    
        # 7. merge heads (최종적으로 (B, T, DIM_MODEL) 로 만드는게 목표이고, 이 과정에서 해드끼리 병합해야함)
        # (B, H, T, HEAD_DIM) -> (B, T, H, HEAD_DIM)
        # contiguous : 메모리 배치를 view 하기 좋게 재정렬하는 과정
        context = context.transpose(1, 2).contiguous().view(B, -1, self.d_model)
    
        # 8. output projection (B, T, DIM_MODEL)
        output = self.W_o(context)
    
        return output

In [34]:
# MHA 테스트
mha = MultiHeadAttention()

mha_example = mha(positional_embed_example, positional_embed_example, positional_embed_example)
print(mha_example.shape)

torch.Size([32, 128, 256])


#### 3. Feed-Forward Network

In [36]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int = D_MODEL, d_ff: int = D_FF, dropout: float = DROPOUT):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [37]:
# Feed-Forward 테스트
ff = FeedForward()

ff_example = ff(mha_example)
print(ff_example.shape)

torch.Size([32, 128, 256])


#### 4. Encoder Layer

In [39]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, d_ff: int = D_FF, dropout: float = DROPOUT):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Self-Attention + residual
        x = self.self_attn(x, x, x, src_mask)
        x = x + self.dropout(x)
        x = self.norm1(x)

        # FFN + residual
        x = self.ff(x)
        x = x + self.dropout(x)
        x = self.norm2(x)

        return x

In [40]:
# Encoder 동작 확인
enc_layer = EncoderLayer()
enc_layer_example = enc_layer(positional_embed_example)

print(enc_layer_example.shape)

torch.Size([32, 128, 256])


#### 5. Decoder Layer

In [42]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, d_ff: int = D_FF, dropout: float = DROPOUT):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, 
        x: torch.Tensor,
        enc_out: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        src_mask: torch.Tensor = None
    ) -> torch.Tensor:
        # 1. Masked self attention
        x = self.self_attn(x, x, x, tgt_mask)
        x = x + self.dropout(x)
        x = self.norm1(x)

        # 2. Cross Attention (Q = Decoder, K = Encoder, V = Encoder)
        x = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = x + self.dropout(x)
        x = self.norm2(x)

        # 3. Feed-Forward
        x = self.ff(x)
        x = x + self.dropout(x)
        x = self.norm3(x)

        return x

In [43]:
# Decoder 동작 확인
dec_layer = DecoderLayer()
tgt_t = torch.randn(2, 8, D_MODEL)
enc_t = torch.randn(2, 10, D_MODEL)

print(f'DecoderLayer output : {dec_layer(tgt_t, enc_t).shape}')  # (2, 8, 256)

DecoderLayer output : torch.Size([2, 8, 256])


#### 6. Encoder

In [45]:
class Encoder(nn.Module):
    def __init__(self, vocab_size: int = VOCAB_SIZE, d_model: int = D_MODEL,
                 n_layers: int = N_LAYERS, n_heads: int = N_HEADS,
                 d_ff: int = D_FF, dropout: float = DROPOUT, max_len: int = MAX_LEN):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pe = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout)
                                        for _ in range(n_layers)])
        self.scale = math.sqrt(d_model)

    def forward(self, src: torch.Tensor, src_mask: torch.Tensor = None) -> torch.Tensor:
        x = self.embedding(src) * self.scale
        x = self.pe(x)

        for layer in self.layers:
            x = layer(x)

        return x

In [46]:
# 테스트
src_batch, tgt_batch = next(iter(train_dl))

encoder_example = Encoder()
encoder_example_output = encoder_example(src_batch)

print(encoder_example_output.shape)

torch.Size([32, 128, 256])


#### 7. Decoder

In [48]:
class Decoder(nn.Module):
    def __init__(self, vocab_size: int = VOCAB_SIZE, d_model: int = D_MODEL,
                 n_layers: int = N_LAYERS, n_heads: int = N_HEADS,
                 d_ff: int = D_FF, dropout: float = DROPOUT, max_len: int = MAX_LEN):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pe = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.scale = math.sqrt(d_model)

    def forward(
        self,
        tgt: torch.Tensor,
        enc_out: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        src_mask: torch.Tensor = None
    ) -> torch.Tensor:
        x = self.embedding(tgt) * self.scale
        x = self.pe(x)

        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, src_mask)

        return x

#### 8. Build Transformer

In [92]:
class Transformer(nn.Module):
    def __init__(
        self,
        vocab_size: int  = VOCAB_SIZE,
        d_model:    int  = D_MODEL,
        n_layers:   int  = N_LAYERS,
        n_heads:    int  = N_HEADS,
        d_ff:       int  = D_FF,
        dropout:    float = DROPOUT,
        max_len:    int  = MAX_LEN,
        pad_id:     int  = PAD_ID,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.encoder = Encoder(vocab_size, d_model, n_layers, n_heads, d_ff, dropout, max_len)
        self.decoder = Decoder(vocab_size, d_model, n_layers, n_heads, d_ff, dropout, max_len)
        self.fc_out  = nn.Linear(d_model, vocab_size, bias=False)
        self.fc_out.weight = self.decoder.embedding.weight
        # Xavier uniform 초기화 (논문 권장)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src: torch.Tensor) -> torch.Tensor:
        """
        PAD_MASK (PAD 위치만 True 로 색인하고 브로드캐스팅)
        (B, SRC_LEN) -> (B, 1, 1, SRC_LEN)
        """
        return (src == self.pad_id).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        """
        CASUAL_MASK | PAD_MASK (TGT에 대한 상삼각행렬 or PAD_MASK)
        (B, TGT_LEN) -> (B, 1, TGT_LEN, TGT_LEN)
        """
        B, T = tgt.shape
        # (B, 1, 1, TGT_LEN)
        pad_mask = (tgt == self.pad_id).unsqueeze(1).unsqueeze(2)
        # (TGT_LEN, TGT_LEN)
        casual_mask = torch.triu(torch.ones(T, T, device=tgt.device), diagonal=1).bool()
        unsqueezed_casual_mask = casual_mask.unsqueeze(0).unsqueeze(0)

        return pad_mask | unsqueezed_casual_mask

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        src_mask, tgt_mask = self.make_src_mask(src), self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, tgt_mask , src_mask)
        return self.fc_out(dec_out)

#### 8. Build the smoke test

In [97]:
model = Transformer().to(DEVICE)

B, T_src, T_tgt = 4, 20, 15

# 특수 토큰 이후 범위의 id 들만 (B, T_src) or (B, T_tgt) dimension 으로 할당
src = torch.randint(4, VOCAB_SIZE, (B, T_src)).to(DEVICE)
tgt = torch.randint(4, VOCAB_SIZE, (B, T_tgt)).to(DEVICE)

with torch.no_grad():
    logits = model(src, tgt)

print(f'logits shape  : {logits.shape}')                    # (4, 15, 16000)
print(f'src_mask shape: {model.make_src_mask(src).shape}')  # (4, 1, 1, 20)
print(f'tgt_mask shape: {model.make_tgt_mask(tgt).shape}')  # (4, 1, 15, 15)

logits shape  : torch.Size([4, 15, 16000])
src_mask shape: torch.Size([4, 1, 1, 20])
tgt_mask shape: torch.Size([4, 1, 15, 15])


#### 9. 파라미터 수 확인

In [106]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'전체 파라미터  : {total:,}')
print(f'학습 가능     : {trainable:,}')
print(f'메모리 추정   : {total * 4 / 1024**2:.1f} MB (fp32)')

print()

for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:<10}: {n:>10,}')

전체 파라미터  : 15,552,512
학습 가능     : 15,552,512
메모리 추정   : 59.3 MB (fp32)

  encoder   :  7,250,944
  decoder   :  8,301,568
  fc_out    :  4,096,000


## 손실함수, 옵티마이저 정의

#### 0. 학습 파라미터 정의

In [114]:
LABEL_SMOOTH = 0.1 # 논문 권장 값

#### 1. Loss function (Label Smoothing Cross Entropy)

In [128]:
class LabelSmoothingLoss(nn.Module):
    def __init__(
        self,
        vocab_size: int = VOCAB_SIZE,
        smoothing: float = LABEL_SMOOTH,
        pad_id: int = PAD_ID
    ):
        super().__init__()
        self.smoothing = smoothing
        self.vocab_size = vocab_size
        self.pad_id = pad_id

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        logits : (B * T, V)
        target : (B * T,)
        """
        B_T, V = logits.shape

        # smooth target 분포 생성
        smooth_val = self.smoothing / (V - 2) # pad, 정답 토큰 재외
        dist = torch.full((B_T, V), smooth_val, device=logits.device) # 모든 위치를 smooth_val 로 filling
        dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing) # 정답 위치에 대해서만 1.0 - self.smoothing
        dist[1, self.pad_id] = 0.0 # pad 위치에 대해서는 예측 목표가 되어선 안되므로 0 으로 초기화

        # PAD 토큰 위치 마스킹
        dist_mask = (target == self.pad_id)
        dist[dist_mask] = 0.0

        log_prob = F.log_softmax(logits, dim=-1)
        loss = - (dist * log_prob).sum(dim=-1)
        return loss[~pad_mask].mean() # pad 위치만 제외하고 평균 반환

#### 2. 옵티마이저 - Noam LR Schedule